# Support quality analytics

> **Synthetic data.** Every row was generated by `scripts/generate_data.py`. It is not real company data.

A walkthrough of the analysis. The logic lives in the SQL files under `sql/`.

**Questions a support / QA manager would ask**
1. Are we meeting SLA, and when do we miss it?
2. Which QA criteria fail most, and who is most at risk of errors?
3. Do QA errors actually affect customers (CSAT, reopens)?
4. Are our QA evaluators scoring consistently?
5. Did the Team C coaching programme work?
6. Which agents should get coaching next?

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
from pipeline import build

pd.set_option('display.width', 180)
con = build()
sql = lambda q: con.sql(q).df()

## 1. Data quality
Issues found in the raw extracts, and how `sql/01_clean.sql` handles each one.

In [2]:
sql('SELECT check_name, rows_affected, action FROM data_quality')

,check_name,rows_affected,action
0,Duplicate rows (same ticket ID loaded twice),316,Removed (one row kept per ticket)
1,Resolved before it was created (bad timestamp),40,"Kept for volume, excluded from time metrics"
2,CSAT outside the 1–5 scale,60,Set to NULL (no valid survey)
3,Agent ID not found in the agent list,25,"Kept, team = Unknown"
4,Channel not recorded,632,"Labelled ""unknown"""


In [3]:
sql('SELECT * FROM headline_kpis')

,tickets,agents,sla_met_pct,median_resolution_hours,avg_handle_min,escalation_pct,reopen_pct,survey_response_pct,csat_pct,avg_qa_score,auto_fail_pct,qa_evaluations
0,63180,80,89.3,15.7,12.4,7.4,6.5,21.9,73.4,92.0,15.2,7304


## 2. SLA and operations

In [4]:
sql('SELECT month, tickets, sla_met_pct, median_resolution_hours, csat_pct FROM monthly_ops')

,month,tickets,sla_met_pct,median_resolution_hours,csat_pct
0,2025-10-01,4857,92.9,13.7,76.9
1,2025-11-01,5940,86.1,18.2,73.1
2,2025-12-01,6663,83.0,20.0,70.7
3,2026-01-01,4990,91.4,14.4,72.9
4,2026-02-01,4508,91.4,14.1,72.0
5,2026-03-01,5140,89.8,15.3,73.2
6,2026-04-01,5022,90.3,14.9,73.0
7,2026-05-01,5057,91.2,14.6,72.7
8,2026-06-01,5117,89.5,15.4,73.0
9,2026-07-01,5302,90.2,15.6,76.4


In [5]:
sql('''SELECT weekday, sum(tickets) AS tickets,
              round(sum(tickets * sla_met_pct) / sum(tickets), 1) AS sla_met_pct
       FROM sla_by_weekday GROUP BY weekday, weekday_num ORDER BY weekday_num''')

,weekday,tickets,sla_met_pct
0,Monday,11639.0,83.6
1,Tuesday,10072.0,88.7
2,Wednesday,10308.0,88.4
3,Thursday,10085.0,88.6
4,Friday,10043.0,88.4
5,Saturday,5512.0,98.6
6,Sunday,5521.0,98.1


In [6]:
sql('SELECT * FROM category_ops')

,category,tickets,avg_handle_min,escalation_pct,reopen_pct,sla_met_pct,csat_pct
0,Order status,13921,6.8,3.4,6.5,90.7,74.5
1,Billing question,12698,10.2,4.2,6.5,90.6,74.7
2,Account access,11277,12.4,5.8,6.5,89.0,73.4
3,Refund request,8210,13.5,6.1,6.6,90.5,73.2
4,Technical issue,7600,20.5,15.8,6.4,86.1,70.8
5,Product question,4997,9.2,2.4,6.0,90.9,78.1
6,Billing dispute,4477,24.1,26.0,7.2,83.3,66.1


SLA falls during the Nov–Dec peak and on Mondays. Both are high-volume periods, so this points to staffing, not agent quality. Billing disputes and technical issues have the highest escalation rates and the lowest CSAT.

## 3. Quality: what fails and who fails it

In [7]:
sql('SELECT * FROM qa_by_criterion')

,criterion,is_critical,weight,fails,fail_rate_pct,share_of_all_fails_pct
0,documentation,False,15,1138,15.6,29.5
1,empathy,False,10,989,13.5,25.6
2,greeting,False,10,559,7.7,14.5
3,accuracy,True,25,461,6.3,11.9
4,policy_compliance,True,20,400,5.5,10.4
5,verification,True,20,315,4.3,8.2


In [8]:
sql('SELECT * FROM qa_by_tenure')

,tenure_band,evaluations,avg_qa_score,auto_fail_pct
0,0-3 months,695,88.2,21.2
1,3-6 months,558,90.1,18.5
2,6+ months,6046,92.6,14.3


## 4. Do QA errors reach the customer?

In [9]:
sql('SELECT * FROM qa_vs_outcomes')

,qa_result,evaluated_tickets,reopen_pct,escalation_pct,csat_responses,csat_pct
0,Critical error found,1113,13.7,9.3,242,52.9
1,No critical error,6191,5.5,6.8,1345,77.2


In [10]:
sql('SELECT * FROM criterion_impact')

,criterion,csat_pct_when_passed,csat_pct_when_failed,reopen_pct_when_passed,reopen_pct_when_failed,csat_responses_when_failed
0,accuracy,76.6,23.1,5.9,18.7,91
1,policy_compliance,74.5,57.8,6.5,11.5,90
2,empathy,75.4,61.0,6.8,6.2,205
3,documentation,74.7,67.8,6.5,8.3,270
4,greeting,73.3,76.2,6.6,8.1,130
5,verification,73.4,76.7,6.6,9.5,73


Accuracy errors are the most damaging to CSAT and reopens. Verification and greeting errors barely move CSAT, because customers do not see them. That is exactly why verification needs QA: it is a risk the customer survey will never reveal.

## 5. Evaluator calibration

In [11]:
sql('SELECT * FROM evaluator_calibration ORDER BY criteria_type, agreement_pct')

,evaluator_id,criteria_type,judgements,agreement_pct,stricter_than_consensus,more_lenient_than_consensus
0,QE03,Critical criteria,360,98.9,3,1
1,QE04,Critical criteria,360,99.2,2,1
2,QE05,Critical criteria,360,99.2,1,2
3,QE01,Critical criteria,360,99.4,2,0
4,QE02,Critical criteria,360,99.4,2,0
5,QE06,Critical criteria,360,99.7,1,0
6,QE04,Soft-skill criteria,359,80.2,67,4
7,QE06,Soft-skill criteria,359,93.6,1,22
8,QE01,Soft-skill criteria,359,98.3,2,4
9,QE05,Soft-skill criteria,359,98.6,1,4


In [12]:
sql('SELECT * FROM evaluator_scoring')

,evaluator_id,evaluations,avg_qa_score,diff_vs_overall
0,QE01,1219,92.5,0.5
1,QE02,1216,92.4,0.4
2,QE03,1253,92.5,0.5
3,QE04,1159,88.0,-4.0
4,QE05,1255,92.7,0.7
5,QE06,1202,93.9,1.9


QE04 fails soft-skill criteria that the majority passes, and scores agents about 4 points lower in live QA. QE06 is lenient on the same criteria. Because tickets are assigned to evaluators at random, an agent's score partly depends on who evaluated them. Recalibrate before using these scores in performance reviews.

## 6. Coaching impact (difference-in-differences)

In [13]:
sql('SELECT * FROM coaching_impact')

,grp,evaluations,qa_score_before,qa_score_after,qa_score_change,auto_fail_pct_before,auto_fail_pct_after,auto_fail_change_pts
0,Team C (coached),741,92.7,94.4,1.8,13.6,11.5,-2.1
1,Other teams,4877,92.6,92.3,-0.2,14.2,14.9,0.7


In [14]:
sql('SELECT * FROM coaching_did')

,qa_score_effect_pts,auto_fail_effect_pts
0,2.0,-2.8


In [15]:
sql('SELECT * FROM coaching_trend')

,quarter,grp,auto_fail_pct,avg_qa_score
0,2025-10-01,Other teams,14.0,92.7
1,2026-01-01,Other teams,14.4,92.5
2,2026-04-01,Other teams,13.9,92.5
3,2026-07-01,Other teams,15.9,92.2
4,2025-10-01,Team C (coached),13.8,92.9
5,2026-01-01,Team C (coached),13.5,92.4
6,2026-04-01,Team C (coached),14.4,93.4
7,2026-07-01,Team C (coached),8.8,95.4


Comparing Team C's change with the change for other teams over the same period removes effects that hit everyone, such as seasonality. The effect builds over time: Q3 2026 shows the biggest gap.

## 7. Agent scorecard (Jul–Sep 2026)

In [16]:
sql('SELECT * FROM agent_scorecard WHERE flag IS NOT NULL')

,agent_id,team,tenure_band,tickets,avg_handle_min,sla_met_pct,reopen_pct,csat_pct,csat_responses,evaluations,avg_qa_score,auto_fail_pct,qa_rank,flag
0,AG011,Team B,3-6 months,188,14.1,89.4,8.0,82.9,35,27,87.8,22.2,74,Coaching priority
1,AG017,Team B,6+ months,183,12.0,86.9,6.6,71.1,38,27,87.6,25.9,75,Coaching priority
2,AG020,Team B,6+ months,207,11.3,93.7,8.7,74.0,50,28,87.5,32.1,76,Coaching priority
3,AG073,Team H,3-6 months,211,14.6,92.9,7.1,76.9,39,28,86.8,25.0,77,Coaching priority
4,AG062,Team G,3-6 months,184,14.2,90.8,6.0,63.4,41,28,86.6,25.0,78,Coaching priority
5,AG046,Team E,6+ months,203,11.4,87.2,7.9,86.7,45,26,86.0,38.5,79,Coaching priority
6,AG044,Team E,3-6 months,199,14.9,91.5,6.5,70.5,44,27,84.1,33.3,80,Coaching priority


## 8. Recommendations
1. **Staff for Mondays and the Nov–Dec peak.** SLA misses follow volume, not agent quality.
2. **Focus QA on accuracy and on agents in their first 3 months.** These are the errors that drive reopens and low CSAT.
3. **Recalibrate QE04 and QE06** on soft-skill criteria before their scores feed into agent reviews.
4. **Roll out the Team C coaching programme** to other teams, starting with the teams of the flagged agents.
5. **Fix data capture.** 1% of tickets have no channel, and some survey and timestamp values are invalid.

**Limitations:** synthetic data, so the size of each effect comes from the generator's settings. The value of this project is the method: the queries, checks and comparisons apply unchanged to real ticket and QA data.